In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import sys
import os

# Add the parent directory so we can import from the 'src' folder
sys.path.append(os.path.abspath(os.path.join('..')))

from src.models.eegnet import EEGNet
from src.data_prep import get_cleaned_epochs
from src.data_prep import get_multi_subject_data
epochs_multi, event_id = get_multi_subject_data(subject_list=[1, 2, 3], training=True)




/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading and Scaling Subject 1...
Loading and Scaling Subject 2...
Loading and Scaling Subject 3...
Not setting metadata
144 matching events found
Applying baseline correction (mode: mean)
Successfully combined 3 subjects.


/Users/elijahakpan/developer/NeuralStream/src/data_prep.py:52: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  combined_epochs = mne.concatenate_epochs(all_epochs)


In [2]:
# Use the modular function we created to get the clean data
epochs, event_id = get_cleaned_epochs(subject_id=1)

# Convert MNE epochs to a NumPy array [Trials, Channels, TimePoints]
X = epochs.get_data() 
y = epochs.events[:, -1] - 769  # Normalize labels to start at 0


In [3]:
import numpy as np

# 1. Get the raw event codes
raw_y = epochs.events[:, -1]

# 2. Identify the unique classes (e.g., 769, 770, 771, 772)
unique_classes = np.unique(raw_y)
print(f"Raw classes found in data: {unique_classes}")

# 3. Create a mapping to 0, 1, 2, 3
# This maps the smallest ID to 0, next to 1, etc.
label_map = {raw_code: i for i, raw_code in enumerate(unique_classes)}
print(f"Mapping labels to: {label_map}")

# 4. Apply the mapping
y = np.array([label_map[code] for code in raw_y])

# 5. Convert to Tensor
y_tensor = torch.tensor(y, dtype=torch.long)

# Now proceed to create your DataLoader...


Raw classes found in data: [1 2 3 4]
Mapping labels to: {np.int64(1): 0, np.int64(2): 1, np.int64(3): 2, np.int64(4): 3}


In [4]:
# Add the '1' dimension for the Convolutional layers
if X.ndim == 3:
    X = X[:, None, :, :]

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

# Create the DataLoader for batching
dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)



In [5]:
# 1. Initialize the model with dynamic dimensions
model = EEGNet(nb_classes=len(event_id), Chans=X.shape[2], Samples=X.shape[-1])

# 2. Setup the "Math" parts
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1) 

# 3. The Training Loop
print("Starting training...")
for epoch in range(50):
    model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()           # Reset the math
        outputs = model(batch_X)        # Make a guess
        loss = criterion(outputs, batch_y) # Calculate the error
        loss.backward()                 # Calculate the fix
        optimizer.step()                # Apply the fix (Learning)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

print("Training Complete!")
print(f"Model Flatten Size: {model.flatten_size}")
%load_ext autoreload
%autoreload 2


Starting training...
Epoch 0, Loss: 1.5119
Epoch 10, Loss: 1.0520
Epoch 20, Loss: 0.6766
Epoch 30, Loss: 0.4425
Epoch 40, Loss: 0.4053
Training Complete!
Model Flatten Size: 1056


In [6]:
model.eval() # Set the model to evaluation mode
correct = 0
total = 0

with torch.no_grad(): # No need to calculate gradients for evaluation
    for batch_X, batch_y in train_loader:
        outputs = model(batch_X)
        _, predicted = torch.max(outputs.data, 1) # Get the highest probability class
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy on the training set: {accuracy:.2f}%')


Accuracy on the training set: 100.00%


In [7]:
# 1. Load the Evaluation data (Set training=False)
epochs_test, _ = get_cleaned_epochs(subject_id=1, training=False)

# 2. Convert to NumPy (This is where your previous error was)
X_test = epochs_test.get_data() 

# 3. Label Mapping (Use the SAME map from your training cell)
y_test = np.array([label_map[code] for code in epochs_test.events[:, -1]])

# 4. Prepare Tensors
if X_test.ndim == 3:
    X_test = X_test[:, None, :, :]

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print(f"Test data loaded: {X_test_tensor.shape[0]} trials.")


Test data loaded: 48 trials.


In [8]:
# 1. Initialize the model with dynamic dimensions
model = EEGNet(nb_classes=len(event_id), Chans=X.shape[2], Samples=X.shape[-1])

# 2. Setup the "Math" parts
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1) 

# 3. The Training Loop
print("Starting training...")
for epoch in range(50):
    model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()           # Reset the math
        outputs = model(batch_X)        # Make a guess
        loss = criterion(outputs, batch_y) # Calculate the error
        loss.backward()                 # Calculate the fix
        optimizer.step()                # Apply the fix (Learning)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

print("Training Complete!")
print(f"Model Flatten Size: {model.flatten_size}")
%load_ext autoreload
%autoreload 2


Starting training...
Epoch 0, Loss: 1.3469
Epoch 10, Loss: 1.0723
Epoch 20, Loss: 0.6429
Epoch 30, Loss: 0.5157
Epoch 40, Loss: 0.4635
Training Complete!
Model Flatten Size: 1056
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
model.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    outputs = model(X_test_tensor)
    _, predicted = torch.max(outputs.data, 1)
    test_total = y_test_tensor.size(0)
    test_correct = (predicted == y_test_tensor).sum().item()

test_accuracy = 100 * test_correct / test_total
print(f'REAL-WORLD ACCURACY (Unseen Session): {test_accuracy:.2f}%')


REAL-WORLD ACCURACY (Unseen Session): 33.33%


In [10]:
# 1. Load data from 3 different people to prevent overfitting
epochs_multi, event_id = get_multi_subject_data(subject_list=[1, 2, 3], training=True)

# 2. Convert to NumPy
X_multi = epochs_multi.get_data()
raw_y_multi = epochs_multi.events[:, -1]

# 3. Create the Label Map
unique_classes = np.unique(raw_y_multi)
label_map = {raw_code: i for i, raw_code in enumerate(unique_classes)}
y_multi = np.array([label_map[code] for code in raw_y_multi])

# 4. Prepare Tensors
if X_multi.ndim == 3:
    X_multi = X_multi[:, None, :, :]

X_train_multi = torch.tensor(X_multi, dtype=torch.float32)
y_train_multi = torch.tensor(y_multi, dtype=torch.long)

# 5. Create DataLoader
train_loader_multi = DataLoader(TensorDataset(X_train_multi, y_train_multi), batch_size=32, shuffle=True)

print(f"Total Training Samples: {X_train_multi.shape[0]}")



Loading and Scaling Subject 1...
Loading and Scaling Subject 2...
Loading and Scaling Subject 3...
Not setting metadata
144 matching events found
Applying baseline correction (mode: mean)
Successfully combined 3 subjects.
Total Training Samples: 144


/Users/elijahakpan/developer/NeuralStream/src/data_prep.py:52: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  combined_epochs = mne.concatenate_epochs(all_epochs)


In [11]:
# Initialize a FRESH model for the multi-subject data
model_multi = EEGNet(nb_classes=len(event_id), Chans=X_multi.shape[2], Samples=X_multi.shape[-1])
optimizer = torch.optim.Adam(model_multi.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1) 

print("Starting Multi-Subject Training...")
for epoch in range(50):
    model_multi.train()
    for batch_X, batch_y in train_loader_multi:
        optimizer.zero_grad()
        outputs = model_multi(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

print("Multi-Subject Training Complete!")


Starting Multi-Subject Training...
Epoch 0, Loss: 1.4012
Epoch 10, Loss: 1.1115
Epoch 20, Loss: 1.0188
Epoch 30, Loss: 0.7597
Epoch 40, Loss: 0.5479
Multi-Subject Training Complete!


In [12]:
model_multi.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    # Use the X_test_tensor you created in the previous section
    outputs = model_multi(X_test_tensor)
    _, predicted = torch.max(outputs.data, 1)
    test_total = y_test_tensor.size(0)
    test_correct = (predicted == y_test_tensor).sum().item()

new_test_accuracy = 100 * test_correct / test_total
print(f'NEW REAL-WORLD ACCURACY: {new_test_accuracy:.2f}%')
print(f'Improvement: {new_test_accuracy - 35.42:.2f}%')


NEW REAL-WORLD ACCURACY: 39.58%
Improvement: 4.16%
